# Scrape J! Archive Episodes by Season

Download episodes from [j-archive.com](https://j-archive.com) and save them to a local DuckDB database.

This notebook scrapes **Season 1** (first 10 episodes only, as a test).
The site's `robots.txt` is checked before any requests, and the `Crawl-delay` is respected.

In [1]:
import logging

from tqdm.notebook import tqdm

from jt3 import save_episode
from jt3.crawler import check_robots, episode_url, fetch_episode, get_season_game_ids

logging.basicConfig(level=logging.INFO)

## Check robots.txt

Verify that j-archive.com allows us to crawl, and read the required delay between requests.

In [2]:
crawl_delay = check_robots()

if crawl_delay is None:
    raise PermissionError("robots.txt disallows crawling — aborting.")

print(f"Crawling allowed. Crawl-delay: {crawl_delay}s")

Crawling allowed. Crawl-delay: 20.0s


## Fetch Season 1 game IDs

Get all episode game IDs for Season 1 from the season page, then take the first 10 (chronological order) as a test.

In [3]:
import time

import requests

# Season page lists newest-first; reverse for chronological order
all_game_ids = get_season_game_ids(42)
all_game_ids.reverse()

print(
    f"Season 42: {len(all_game_ids)} episodes total, fetching first {len(all_game_ids)}"
)

episodes = []

for i, gid in enumerate(pbar := tqdm(all_game_ids, desc="Fetching episodes")):
    if i > 0 and crawl_delay and crawl_delay > 0:
        time.sleep(crawl_delay)

    url = episode_url(gid)
    pbar.set_postfix(game_id=gid)

    try:
        ep = fetch_episode(url)
    except requests.HTTPError as exc:
        print(f"  Skipping game_id={gid}: {exc}")
        continue

    save_episode(ep)
    episodes.append(ep)

print(f"\nSaved {len(episodes)} episodes to DuckDB.")

Season 42: 153 episodes total, fetching first 153


Fetching episodes:   0%|          | 0/153 [00:00<?, ?it/s]


Saved 153 episodes to DuckDB.


## Summary

List all episodes that were saved.

In [4]:
for ep in episodes:
    print(
        f"game_id={ep.game_id:>5}  "
        f"show=#{ep.show_number or '???':>5}  "
        f"aired={ep.air_date or 'unknown'}  "
        f"clues={sum(len(cat.clues) for rnd in ep.rounds for cat in rnd.categories)}"
    )

game_id= 9263  show=# 9386  aired=2025-09-08  clues=61
game_id= 9264  show=# 9387  aired=2025-09-09  clues=61
game_id= 9265  show=# 9388  aired=2025-09-10  clues=61
game_id= 9266  show=# 9389  aired=2025-09-11  clues=61
game_id= 9267  show=# 9390  aired=2025-09-12  clues=61
game_id= 9268  show=# 9391  aired=2025-09-15  clues=61
game_id= 9269  show=# 9392  aired=2025-09-16  clues=61
game_id= 9270  show=# 9393  aired=2025-09-17  clues=61
game_id= 9271  show=# 9394  aired=2025-09-18  clues=61
game_id= 9272  show=# 9395  aired=2025-09-19  clues=61
game_id= 9273  show=# 9396  aired=2025-09-22  clues=61
game_id= 9274  show=# 9397  aired=2025-09-23  clues=58
game_id= 9275  show=# 9398  aired=2025-09-24  clues=61
game_id= 9276  show=# 9399  aired=2025-09-25  clues=61
game_id= 9277  show=# 9400  aired=2025-09-26  clues=61
game_id= 9278  show=# 9401  aired=2025-09-29  clues=61
game_id= 9279  show=# 9402  aired=2025-09-30  clues=61
game_id= 9280  show=# 9403  aired=2025-10-01  clues=61
game_id= 9